# 27 · Ativos, engajamento e churn por canal de aquisição

Pergunta: como é a relação de ativos, inativos (churn) e engajados por canal de aquisição?

Três fatores, sempre no corte de 30/09/2026:
- **ativos**: assinatura ativa no corte;
- **churn**: assinaturas encerradas (acumulado) e taxa mensal (perdas ÷ ativas no início do mês);
- **engajamento** (só assinantes ativos), pela função `niveis_engajamento` de `src/engajamento.py`:
  alto = treinou nos últimos 7 dias; médio = treinou no mês, mas não na última semana;
  baixo = 30 a 44 dias sem treinar; possível churn = 45 dias ou mais.

In [1]:
import sys
import pandas as pd
from pathlib import Path
sys.path.insert(0, "../src")
from engajamento import niveis_engajamento, NIVEIS

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")

usuarios = pd.read_parquet(P / "usuarios_limpa.parquet", columns=["user_id", "canal_aquisicao", "data_cadastro", "cadastro_antes_do_log"])
assin = pd.read_parquet(P / "assinaturas_limpa.parquet")
painel = pd.read_parquet(P / "assinaturas_mes.parquet", columns=["mes", "user_id", "plano", "ativa_inicio_mes", "perdida_no_mes", "iniciou_no_mes"])
ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data"])
print(len(usuarios), "contas |", len(assin), "assinaturas |", len(painel), "linhas do painel |", len(ativ), "treinos")

63929 contas | 32583 assinaturas | 327771 linhas do painel | 1252488 treinos


## 0. Níveis de engajamento dos assinantes ativos (base para todas as perguntas)

In [2]:
ativos = assin[assin.ativa_no_corte]
eng = niveis_engajamento(ativ, ativos.user_id, CORTE, inicio=ativos.set_index("user_id").inicio)
print("assinantes ativos:", len(ativos), "-> classificados:", len(eng))
t = eng.nivel.value_counts().reindex(NIVEIS).rename("ativos").to_frame()
t["%"] = t.ativos / t.ativos.sum() * 100
print(t)
print()
print("casos que a definição original não cobria:")
print("  médio com 5+ treinos no mês (sem treino na última semana):", int(eng.medio_5_mais.sum()))
print("  médio com 1 a 4 treinos no mês:", int(((eng.nivel == NIVEIS[1]) & ~eng.medio_5_mais & ~eng.nunca_treinou).sum()))
print("  nunca treinou desde 01/2026 (dias contados do início da assinatura):", int(eng.nunca_treinou.sum()))
print(eng[eng.nunca_treinou].nivel.value_counts().reindex(NIVEIS).fillna(0).astype(int).to_dict())
print()
print("treinos nos últimos 30 dias por nível (mediana):", eng.groupby("nivel").treinos_30d.median().to_dict())
eng.to_parquet(P / "engajamento_corte.parquet", index=False)

assinantes ativos: 14753 -> classificados: 14753
                   ativos    %
nivel                         
1. alto             10689 72.5
2. médio             2371 16.1
3. baixo              365  2.5
4. possível churn    1328  9.0

casos que a definição original não cobria:
  médio com 5+ treinos no mês (sem treino na última semana): 977
  médio com 1 a 4 treinos no mês: 1389
  nunca treinou desde 01/2026 (dias contados do início da assinatura): 176
{'1. alto': 0, '2. médio': 5, '3. baixo': 0, '4. possível churn': 171}

treinos nos últimos 30 dias por nível (mediana): {'1. alto': 11.0, '2. médio': 3.0, '3. baixo': 0.0, '4. possível churn': 0.0}


## 1. Ativos e churn acumulado por canal

In [3]:
b = usuarios.merge(assin[["user_id", "plano_atual", "ativa_no_corte", "inicio", "inicio_antes_do_log"]], on="user_id", how="left", validate="one_to_one")
print("contas:", len(usuarios), "-> depois da junção:", len(b))
b["assinou"] = b.plano_atual.notna()
b["ativo"] = b.ativa_no_corte == True
b["saiu"] = b.assinou & ~b.ativo

def resumo(d):
    t = d.groupby("canal_aquisicao").agg(contas=("user_id", "size"), ja_assinou=("assinou", "sum"), ativos=("ativo", "sum"), sairam=("saiu", "sum"))
    t.loc["total"] = t.sum()
    t["% das contas"] = t.contas / t.contas["total"] * 100
    t["% dos ativos"] = t.ativos / t.ativos["total"] * 100
    t["% dos que saíram"] = t.sairam / t.sairam["total"] * 100
    t["% assinou"] = t.ja_assinou / t.contas * 100
    t["% saiu (de quem assinou)"] = t.sairam / t.ja_assinou * 100
    return t.sort_values("contas", ascending=False)

print("todas as contas"); geral = resumo(b); print(geral)
print()
# Conversão e saída só para cadastros desde 07/2024 (antes disso faltam as contas grátis)
print("cadastro desde 07/2024:", (~b.cadastro_antes_do_log).sum(), "contas")
print(resumo(b[~b.cadastro_antes_do_log])[["contas", "ja_assinou", "ativos", "sairam", "% assinou", "% saiu (de quem assinou)"]])

contas: 63929 -> depois da junção: 63929
todas as contas
                      contas  ja_assinou  ativos  sairam  % das contas  % dos ativos  % dos que saíram  % assinou  % saiu (de quem assinou)
canal_aquisicao                                                                                                                            
total                  63929       32583   14753   17830         100.0         100.0             100.0       51.0                      54.7
organico               18256       10058    4634    5424          28.6          31.4              30.4       55.1                      53.9
influenciador          13851        5734    2495    3239          21.7          16.9              18.2       41.4                      56.5
meta_ads               11783        5681    2425    3256          18.4          16.4              18.3       48.2                      57.3
indicacao               7993        4635    2228    2407          12.5          15.1              13.5 

In [4]:
a = b[b.assinou]
print("plano de quem já assinou, por canal (% da linha) |", len(a), "assinaturas")
print(pd.crosstab(a.canal_aquisicao, a.plano_atual, normalize="index", margins=True) * 100)
print()
print("% que saiu, por canal e plano")
print(a.pivot_table(index="canal_aquisicao", columns="plano_atual", values="saiu", aggfunc="mean", margins=True) * 100)
print()
print("contas por ano de cadastro e canal (% da coluna)")
print(pd.crosstab(b.canal_aquisicao, b.data_cadastro.dt.year, normalize="columns") * 100)

plano de quem já assinou, por canal (% da linha) | 32583 assinaturas
plano_atual           anual  mensal
canal_aquisicao                    
google_ads             41.3    58.7
indicacao              40.7    59.3
influenciador          30.7    69.3
meta_ads               42.0    58.0
organico               42.6    57.4
parceria_corporativa   42.3    57.7
All                    40.0    60.0

% que saiu, por canal e plano
plano_atual           anual  mensal  All
canal_aquisicao                         
google_ads             36.1    69.9 55.9
indicacao              30.8    66.4 51.9
influenciador          34.4    66.3 56.5
meta_ads               38.1    71.2 57.3
organico               35.3    67.8 53.9
parceria_corporativa   33.1    66.0 52.1
All                    34.9    67.9 54.7

contas por ano de cadastro e canal (% da coluna)
data_cadastro         2020  2021  2022  2023  2024  2025  2026
canal_aquisicao                                               
google_ads            10.8  13.

## 2. Taxa de churn mensal por canal

Perdas do mês ÷ assinaturas ativas no início do mês, somadas no período (jul/2024 fica fora).

In [5]:
p = painel.merge(usuarios[["user_id", "canal_aquisicao"]], on="user_id", how="left", validate="many_to_one")
print("painel:", len(painel), "-> depois da junção:", len(p), "| sem canal:", p.canal_aquisicao.isna().sum())
p = p[p.mes >= "2024-08-01"]
print("sem jul/2024:", len(p))
p["ano"] = p.mes.dt.year

def churn(d, por):
    g = d.groupby(por).agg(ativas_inicio=("ativa_inicio_mes", "sum"), perdas=("perdida_no_mes", "sum"))
    g["churn mensal %"] = g.perdas / g.ativas_inicio * 100
    return g

print(); print("2026 (jan a set), todos os planos"); c26 = churn(p[p.ano == 2026], "canal_aquisicao"); print(c26.sort_values("churn mensal %"))
print("total 2026:", f"{p[p.ano == 2026].perdida_no_mes.sum() / p[p.ano == 2026].ativa_inicio_mes.sum():.2%}")
print(); print("churn mensal % por canal e ano")
print(churn(p, ["canal_aquisicao", "ano"])["churn mensal %"].unstack())
print(); print("churn mensal % em 2026 por canal e plano")
cp = churn(p[p.ano == 2026], ["canal_aquisicao", "plano"])
print(cp["churn mensal %"].unstack()); print(cp.perdas.unstack())
print(); print("perdas em 2026 por canal (% do total de perdas) contra % das ativas no início do mês")
x = c26.assign(**{"% das perdas": c26.perdas / c26.perdas.sum() * 100, "% da base ativa": c26.ativas_inicio / c26.ativas_inicio.sum() * 100})
print(x[["perdas", "% das perdas", "% da base ativa"]].sort_values("perdas", ascending=False))

painel: 327771 -> depois da junção: 327771 | sem canal: 0
sem jul/2024: 321927

2026 (jan a set), todos os planos
                      ativas_inicio  perdas  churn mensal %
canal_aquisicao                                            
indicacao                     19261    1128             5.9
parceria_corporativa          12635     746             5.9
organico                      41037    2558             6.2
google_ads                    13850     888             6.4
meta_ads                      21695    1436             6.6
influenciador                 17369    1564             9.0
total 2026: 6.61%

churn mensal % por canal e ano
ano                   2024  2025  2026
canal_aquisicao                       
google_ads             4.6   5.4   6.4
indicacao              4.0   5.2   5.9
influenciador          6.7   7.5   9.0
meta_ads               5.1   5.6   6.6
organico               4.4   5.4   6.2
parceria_corporativa   3.8   5.2   5.9

churn mensal % em 2026 por canal e plano
pl

In [6]:
# Sobrevivência simples: das assinaturas iniciadas de 07/2024 a 09/2025 (12+ meses até o corte), % ainda ativa
c = a[(a.inicio >= "2024-07-01") & (a.inicio <= "2025-09-30")]
print("assinaturas iniciadas de 07/2024 a 09/2025:", len(c))
print("% ainda ativa no corte, por canal e plano")
print(c.pivot_table(index="canal_aquisicao", columns="plano_atual", values="ativo", aggfunc="mean", margins=True) * 100)
print(pd.crosstab(c.canal_aquisicao, c.plano_atual, margins=True))

assinaturas iniciadas de 07/2024 a 09/2025: 14471
% ainda ativa no corte, por canal e plano
plano_atual           anual  mensal  All
canal_aquisicao                         
google_ads             51.0    15.1 29.5
indicacao              51.3    18.8 31.0
influenciador          38.0     9.9 18.0
meta_ads               45.5    16.7 28.4
organico               46.9    17.0 29.6
parceria_corporativa   51.0    19.6 32.4
All                    47.3    16.2 28.3
plano_atual           anual  mensal    All
canal_aquisicao                           
google_ads              667     988   1655
indicacao               811    1360   2171
influenciador           539    1336   1875
meta_ads               1063    1561   2624
organico               1987    2751   4738
parceria_corporativa    572     836   1408
All                    5639    8832  14471


## 3. Engajamento dos assinantes ativos por canal

In [7]:
e = eng.merge(b[["user_id", "canal_aquisicao", "plano_atual"]], on="user_id", how="left", validate="one_to_one")
print("ativos classificados:", len(eng), "-> depois da junção:", len(e))
n = pd.crosstab(e.canal_aquisicao, e.nivel, margins=True, margins_name="total")[NIVEIS + ["total"]]
pc = pd.crosstab(e.canal_aquisicao, e.nivel, normalize="index", margins=True, margins_name="total")[NIVEIS] * 100
print(n); print(); print(pc.sort_values("1. alto", ascending=False))
print()
print("treinos nos últimos 30 dias por ativo (média e mediana)")
print(e.groupby("canal_aquisicao").treinos_30d.agg(["mean", "median"]))
print()
for pl in ["mensal", "anual"]:
    d = e[e.plano_atual == pl]
    print(f"plano {pl}: {len(d)} ativos (% da linha)")
    print(pd.crosstab(d.canal_aquisicao, d.nivel, normalize="index", margins=True, margins_name="total")[NIVEIS] * 100); print()

final = geral.drop(index="total")[["contas", "ja_assinou", "ativos", "sairam", "% assinou", "% saiu (de quem assinou)"]].join(c26[["churn mensal %"]].rename(columns={"churn mensal %": "churn mensal 2026 %"})).join(n.drop(index="total")[NIVEIS]).join(pc.add_prefix("% "))
final.to_csv(S / "canal_aquisicao_ativos_engajamento_churn.csv", encoding="utf-8-sig")
print(final.T)

ativos classificados: 14753 -> depois da junção: 14753
nivel                 1. alto  2. médio  3. baixo  4. possível churn  total
canal_aquisicao                                                            
google_ads               1093       242        39                158   1532
indicacao                1624       360        52                192   2228
influenciador            1854       381        58                202   2495
meta_ads                 1701       419        65                240   2425
organico                 3363       743       111                417   4634
parceria_corporativa     1054       226        40                119   1439
total                   10689      2371       365               1328  14753

nivel                 1. alto  2. médio  3. baixo  4. possível churn
canal_aquisicao                                                     
influenciador            74.3      15.3       2.3                8.1
parceria_corporativa     73.2      15.7       2.8    

nivel                 1. alto  2. médio  3. baixo  4. possível churn
canal_aquisicao                                                     
google_ads               60.5      18.4       3.9               17.2
indicacao                64.0      18.0       3.4               14.6
influenciador            59.1      18.6       4.8               17.5
meta_ads                 60.2      19.5       4.1               16.3
organico                 63.2      18.3       3.7               14.8
parceria_corporativa     64.8      17.0       4.4               13.9
total                    62.1      18.4       4.0               15.6

canal_aquisicao           organico  influenciador  meta_ads  indicacao  google_ads  parceria_corporativa
contas                    18,256.0       13,851.0  11,783.0    7,993.0     6,701.0               5,345.0
ja_assinou                10,058.0        5,734.0   5,681.0    4,635.0     3,473.0               3,002.0
ativos                     4,634.0        2,495.0   2,425.0    